[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/feature/notebook-rebuild/notebooks/case_studies/data_analysis_assistant/langgraph.ipynb)

# Data-analysis assistant — LangGraph

A four-node graph inspects, selects, executes and reports the same allowlisted analysis as the plain baseline. Mock runtime is under one minute on CPU; local inference is practical.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "langgraph==1.2.9", "pydantic==2.12.5"],
        check=True,
    )

In [ ]:
import csv
import json
from pathlib import Path
from typing import ClassVar

from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, ConfigDict

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import DeterministicMockClient  # noqa: E402
from agentic_tutorial.models.providers.fixtures import ScriptedScenarioFixture  # noqa: E402
from agentic_tutorial.schemas import Message, MessageRole  # noqa: E402
from agentic_tutorial.tools import AnalysisRequest, file_sha256, summarise_reduction  # noqa: E402

data_path = ROOT / "data/data_analysis_assistant/household_waste.csv"
expected = json.loads((ROOT / "data/data_analysis_assistant/expected_summary.json").read_text())
fixture = json.loads((ROOT / "data/data_analysis_assistant/case_mock.json").read_text())

## Graph and deterministic execution boundary
The model never emits code or computed values. Conditional routing rejects unknown tools before execution; schema and oracle mismatches stop before reporting.

In [ ]:
class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")


class Decision(Strict):
    schema_id: ClassVar[str] = "analysis.decision.v1"
    tool: str
    group_column: str
    before_column: str
    after_column: str


class Report(Strict):
    schema_id: ClassVar[str] = "analysis.report.v1"
    report: str
    groups: tuple[str, ...]


def fresh_model():
    return DeterministicMockClient(ScriptedScenarioFixture.model_validate(fixture))


def user(text):
    return Message(role=MessageRole.USER, content=text)


async def propose(client, schema, text):
    response = await client.generate([user(text)], response_schema=schema)
    return schema.model_validate(response.structured_output)


def build_graph():
    client = fresh_model()

    def inspect_node(state):
        with data_path.open(newline="", encoding="utf-8") as handle:
            rows = list(csv.DictReader(handle))
        return {
            "rows": len(rows),
            "columns": tuple(rows[0]) if rows else (),
            "trace": [{"event": "inspect"}],
            "model_calls": 0,
        }

    async def decide_node(state):
        decision = await propose(
            client, Decision, f"Choose permitted analysis for {state['columns']}"
        )
        return {
            **state,
            "decision": decision,
            "model_calls": 1,
            "trace": [*state["trace"], {"event": "decision", "tool": decision.tool}],
        }

    def execute_node(state):
        request = AnalysisRequest(
            group_column=state["decision"].group_column,
            before_column=state["decision"].before_column,
            after_column=state["decision"].after_column,
        )
        result = summarise_reduction(data_path, request)
        valid = result == expected["mean_reduction_kg"]
        return {
            **state,
            "result": result,
            "valid": valid,
            "trace": [*state["trace"], {"event": "execute_and_validate", "valid": valid}],
        }

    async def report_node(state):
        report = await propose(client, Report, f"Report only {state['result']}")
        groups_valid = set(report.groups) == set(state["result"])
        return {
            **state,
            "report": report,
            "outcome": "supported_report" if groups_valid else "fallback",
            "model_calls": 2,
            "termination": "criteria_met" if groups_valid else "validation_failed",
            "trace": [*state["trace"], {"event": "report_validation", "valid": groups_valid}],
        }

    graph = StateGraph(dict)
    graph.add_node("inspect", inspect_node)
    graph.add_node("decide", decide_node)
    graph.add_node("execute", execute_node)
    graph.add_node("report", report_node)
    graph.add_edge(START, "inspect")
    graph.add_edge("inspect", "decide")
    graph.add_conditional_edges(
        "decide",
        lambda s: "execute" if s["decision"].tool == "mean_reduction_by_intervention" else END,
        {"execute": "execute", END: END},
    )
    graph.add_conditional_edges(
        "execute", lambda s: "report" if s["valid"] else END, {"report": "report", END: END}
    )
    graph.add_edge("report", END)
    return graph.compile()


first = await build_graph().ainvoke({})
second = await build_graph().ainvoke({})
evaluation = {
    "component": first["result"] == expected["mean_reduction_kg"],
    "trajectory": len(first["trace"]) == 4 and first["model_calls"] <= 2,
    "task": first["outcome"] == "supported_report",
    "safety": first["decision"].tool == "mean_reduction_by_intervention",
    "repeated_run": first == second,
}
assert all(evaluation.values()), evaluation
{
    "evaluation": evaluation,
    "trace": first["trace"],
    "outputs": first["result"],
    "provenance": {"sha256": file_sha256(data_path)},
    "fallback": "invalid route or result ends before report",
}

## Limitation
Graph control flow cannot make this descriptive dataset causal; arbitrary model-authored analysis remains deliberately unsupported.